# Deteccion de Muelas del Juicio — Entrega 2: Dataset en PyTorch

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Dataset:** DENTEX — MICCAI 2023 ([arXiv:2305.19112](https://arxiv.org/abs/2305.19112))

Notebook independiente. Funciona tanto en Google Colab como en local.  
El dataset se descarga automáticamente desde Hugging Face si no existe.

---

| Seccion | Contenido |
|---------|-----------|
| 1 | Setup: entorno, paths y dependencias |
| 2 | Descarga del dataset desde Hugging Face |
| 3 | Reconstruccion del split YOLO |
| 4 | Clase Dataset de PyTorch |
| 5 | DataLoaders train / val / test |
| 6 | Preprocesamiento y normalizacion |
| 7 | Data augmentation + visualizacion |
| 8 | Verificacion final del batch |
| 9 | Exportar CSVs de splits para GitHub |
| 10 | Estrategias de entrenamiento evaluadas |

## Seccion 1 — Setup

In [ ]:
import subprocess, sys, json, shutil, random, warnings, os
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Detectar entorno: Colab o local ──────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Entorno: {"Google Colab" if IN_COLAB else "Local"}')

# ── Paths segun entorno ───────────────────────────────────────────
if IN_COLAB:
    # En Colab usamos /content/ — no depende de Drive
    BASE_DIR    = Path('/content')
    DATA_DIR    = BASE_DIR / 'data'
else:
    # En local usamos la raiz del repo (este notebook esta en dev/)
    BASE_DIR    = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent
    # Si cwd es la raiz del repo directamente
    if not (BASE_DIR / 'data').exists() and (Path.cwd() / 'data').exists():
        BASE_DIR = Path.cwd()
    DATA_DIR    = BASE_DIR / 'data'

DATASET_DIR = DATA_DIR / 'raw' / 'dentex_raw'
YOLO_DIR    = DATA_DIR / 'processed' / 'yolo_dataset'
SPLITS_DIR  = DATA_DIR                          # CSVs van en data/ para GitHub
OUTPUTS_DIR = DATA_DIR / 'outputs'             # Graficos y logs

for d in [DATASET_DIR, YOLO_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'BASE_DIR:    {BASE_DIR}')
print(f'DATA_DIR:    {DATA_DIR}')
print(f'DATASET_DIR: {DATASET_DIR}')
print(f'YOLO_DIR:    {YOLO_DIR}')

In [ ]:
# GPU check
import torch

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print('GPU detectada')
    else:
        print('Sin GPU — para entrenamiento ir a Entorno de ejecucion > GPU T4')
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('Sin GPU — OK para analisis exploratorio')

print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pip install ultralytics scikit-learn huggingface_hub -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'Device:  {"cuda" if torch.cuda.is_available() else "cpu"}')
print('Dependencias listas')

In [ ]:
def is_real_file(p):
    """True si el archivo existe y no es symlink roto."""
    try:
        return p.resolve().exists() and p.stat().st_size > 0
    except (OSError, FileNotFoundError):
        return False

def log(msg, level='INFO'):
    icons = {'INFO': '[INFO]', 'OK': '[OK]  ', 'WARN': '[WARN]', 'ERR': '[ERR] ', 'DATA': '[DATA]'}
    print(f'{icons.get(level, "[INFO]")} {msg}')

## Seccion 2 — Descarga del dataset desde Hugging Face

El dataset se descarga automaticamente desde Hugging Face si no existe en disco.  
No se requiere Drive ni configuracion adicional.

In [ ]:
from huggingface_hub import snapshot_download

json_files = []
if DATASET_DIR.exists():
    json_files = [f for f in DATASET_DIR.rglob('*.json') if is_real_file(f)]

if json_files:
    log(f'Dataset encontrado — {len(json_files)} JSONs validos', 'OK')
else:
    log('Dataset no encontrado — descargando desde Hugging Face...', 'WARN')
    log('Tamanio estimado: ~3 GB — puede tardar varios minutos', 'INFO')
    snapshot_download(
        repo_id='ibrahimhamamci/DENTEX',
        repo_type='dataset',
        local_dir=str(DATASET_DIR),
        ignore_patterns=['*.git*'],
    )
    json_files = [f for f in DATASET_DIR.rglob('*.json') if is_real_file(f)]
    log(f'Descarga completada — {len(json_files)} JSONs', 'OK')

# Descomprimir validation_data.zip si existe
val_zip = DATASET_DIR / 'DENTEX' / 'validation_data.zip'
val_dir = DATASET_DIR / 'DENTEX' / 'validation_data'
if val_zip.exists() and is_real_file(val_zip) and not val_dir.exists():
    import zipfile
    log('Descomprimiendo validation_data.zip...', 'INFO')
    with zipfile.ZipFile(val_zip, 'r') as z:
        z.extractall(val_zip.parent)
    log('Validacion descomprimida', 'OK')

# Detectar JSON principal
TRAIN_JSON = None
for j in json_files:
    name = j.name.lower()
    if 'disease' in name or 'diagnosis' in name:
        TRAIN_JSON = j
        break

if TRAIN_JSON is None:
    raise FileNotFoundError('JSON dataset (c) no encontrado. Revisa la descarga.')

log(f'JSON principal: {TRAIN_JSON.name}', 'OK')

# Detectar carpeta de imagenes
IMG_DIR_C = None
for candidate in [TRAIN_JSON.parent / 'xrays', TRAIN_JSON.parent.parent / 'xrays']:
    if candidate.exists() and candidate.is_dir():
        IMG_DIR_C = candidate
        break
if IMG_DIR_C is None:
    IMG_DIR_C = TRAIN_JSON.parent / 'xrays'

imgs_reales = [p for p in list(IMG_DIR_C.glob('*.png')) + list(IMG_DIR_C.glob('*.jpg'))
               if is_real_file(p)]
log(f'Imagenes en disco: {len(imgs_reales)}', 'OK')

In [ ]:
# Cargar JSON y construir wisdom_c
with open(TRAIN_JSON) as f:
    data_c = json.load(f)

id_to_image_c = {img['id']: img for img in data_c['images']}

wisdom_c = defaultdict(list)
for ann in data_c['annotations']:
    if ann.get('category_id_2') == 7:
        wisdom_c[ann['image_id']].append(ann)

def classify_image(img_id):
    return 'impacted' if any(
        a.get('category_id_3') == 0 for a in wisdom_c[img_id]
    ) else 'erupted'

img_labels_c = {img_id: classify_image(img_id) for img_id in wisdom_c}
label_counts  = Counter(img_labels_c.values())
total_c       = sum(label_counts.values())

log(f'Imagenes con muela del juicio: {total_c}', 'DATA')
log(f'  erupted:  {label_counts["erupted"]} ({label_counts["erupted"]/total_c*100:.1f}%)', 'DATA')
log(f'  impacted: {label_counts["impacted"]} ({label_counts["impacted"]/total_c*100:.1f}%)', 'DATA')
log(f'  ratio:    {max(label_counts.values())/min(label_counts.values()):.2f}x', 'DATA')

## Seccion 3 — Reconstruccion del split YOLO

Split estratificado 70/5/25 con seed=42.  
Si ya existe en disco lo reutiliza. Si no, lo genera desde cero.

In [ ]:
def verificar_split():
    if not YOLO_DIR.exists():
        return False, 'YOLO_DIR no existe'
    for split in ['train', 'val', 'test']:
        img_dir = YOLO_DIR / 'images' / split
        if not img_dir.exists() or len(list(img_dir.glob('*.*'))) == 0:
            return False, f'Split {split} vacio o inexistente'
    return True, 'OK'

ok, motivo = verificar_split()

if ok:
    counts = {s: len(list((YOLO_DIR / 'images' / s).glob('*.*'))) for s in ['train','val','test']}
    log(f'Split YOLO encontrado — train:{counts["train"]} val:{counts["val"]} test:{counts["test"]}', 'OK')
else:
    log(f'Split no encontrado ({motivo}) — generando...', 'WARN')

    img_ids_c = list(wisdom_c.keys())
    labels_c  = [classify_image(i) for i in img_ids_c]

    ids_train, ids_tmp, _, y_tmp = train_test_split(
        img_ids_c, labels_c, test_size=0.30, stratify=labels_c, random_state=42)
    ids_val, ids_test, _, _ = train_test_split(
        ids_tmp, y_tmp, test_size=0.833, stratify=y_tmp, random_state=42)

    splits_dict = {'train': ids_train, 'val': ids_val, 'test': ids_test}
    errors = 0

    for split, ids in splits_dict.items():
        (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

        for img_id in ids:
            info     = id_to_image_c[img_id]
            src_path = IMG_DIR_C / Path(info['file_name']).name
            if not is_real_file(src_path):
                errors += 1
                continue
            pil  = Image.open(src_path)
            W, H = pil.size
            shutil.copy2(src_path, YOLO_DIR / 'images' / split / src_path.name)
            lines = []
            for ann in wisdom_c[img_id]:
                x, y, bw, bh = ann['bbox']
                xc  = min(max((x + bw/2) / W, 0.0), 1.0)
                yc  = min(max((y + bh/2) / H, 0.0), 1.0)
                wn  = min(max(bw / W, 0.0), 1.0)
                hn  = min(max(bh / H, 0.0), 1.0)
                cls = 1 if ann.get('category_id_3') == 0 else 0
                lines.append(f'{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            lbl_path = YOLO_DIR / 'labels' / split / (src_path.stem + '.txt')
            lbl_path.write_text('\n'.join(lines))

    yaml_content = f"""path: {YOLO_DIR}
train: images/train
val:   images/val
test:  images/test
nc: 2
names:
  0: erupted
  1: impacted
"""
    (YOLO_DIR / 'dataset.yaml').write_text(yaml_content)
    log(f'Split generado{f" ({errors} imgs no encontradas)" if errors else ""}', 'OK')

# Log detallado
log('Estadisticas del split:', 'DATA')
for split in ['train', 'val', 'test']:
    n = len(list((YOLO_DIR / 'images' / split).glob('*.*')))
    c0, c1 = 0, 0
    for lbl in (YOLO_DIR / 'labels' / split).glob('*.txt'):
        for line in lbl.read_text().strip().split('\n'):
            if line:
                if int(line.split()[0]) == 0: c0 += 1
                else: c1 += 1
    log(f'  {split:5s}: {n} imgs | {c0+c1} boxes (erupted={c0}, impacted={c1})', 'DATA')

## Seccion 4 — Clase Dataset de PyTorch

Implementamos `DentexDataset` heredando de `torch.utils.data.Dataset`.

**Por que no ImageFolder:**  
`ImageFolder` asume una imagen = una etiqueta global (clasificacion).  
Nuestro problema es deteccion de objetos — cada imagen puede tener multiples bounding boxes con distintas clases.  
Necesitamos una clase Dataset personalizada.

In [ ]:
class DentexDataset(Dataset):
    """
    Dataset PyTorch para DENTEX — deteccion de muelas del juicio.

    Cada item retorna:
        image  (Tensor): imagen normalizada [C, H, W]
        target (dict):   boxes [N,4] en formato YOLO normalizado,
                         labels [N] (0=erupted, 1=impacted),
                         image_id (str)
    """

    CLASS_NAMES  = {0: 'erupted', 1: 'impacted'}
    CLASS_COLORS = {0: '#44AA44', 1: '#FF4444'}

    def __init__(self, split, yolo_dir, transform=None):
        assert split in ('train', 'val', 'test')
        self.split     = split
        self.yolo_dir  = Path(yolo_dir)
        self.transform = transform
        self.img_dir   = self.yolo_dir / 'images' / split
        self.lbl_dir   = self.yolo_dir / 'labels' / split

        self.samples = []
        all_imgs = sorted(self.img_dir.glob('*.png')) + sorted(self.img_dir.glob('*.jpg'))
        for img_path in all_imgs:
            lbl_path = self.lbl_dir / (img_path.stem + '.txt')
            if lbl_path.exists():
                self.samples.append((img_path, lbl_path))

        self.class_counts = Counter()
        for _, lbl_path in self.samples:
            for line in lbl_path.read_text().strip().split('\n'):
                if line:
                    self.class_counts[int(line.split()[0])] += 1

        log(f'DentexDataset [{split}]: {len(self.samples)} imagenes | '
            f'erupted={self.class_counts[0]} impacted={self.class_counts[1]}', 'OK')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        image = Image.open(img_path).convert('RGB')

        boxes, labels = [], []
        for line in lbl_path.read_text().strip().split('\n'):
            if not line:
                continue
            parts = line.split()
            labels.append(int(parts[0]))
            boxes.append(list(map(float, parts[1:])))

        if self.transform:
            image = self.transform(image)

        target = {
            'boxes':    torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4)),
            'labels':   torch.tensor(labels, dtype=torch.long)    if labels else torch.zeros(0, dtype=torch.long),
            'image_id': img_path.stem,
        }
        return image, target

    def get_image_raw(self, idx):
        """Imagen PIL sin transformaciones — para visualizacion."""
        img_path, _ = self.samples[idx]
        return Image.open(img_path).convert('RGB')

log('Clase DentexDataset definida', 'OK')

## Seccion 5 — DataLoaders

- `batch_size=8` — imagenes grandes (~2800px) que se redimensionan a 640px
- `shuffle=True` solo en train — val y test deben ser deterministas
- `num_workers=2` — carga paralela
- `collate_fn` personalizado — cada imagen tiene distinto numero de boxes

In [ ]:
BASE_TRANSFORM = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])

def collate_fn(batch):
    images  = torch.stack([item[0] for item in batch], dim=0)
    targets = [item[1] for item in batch]
    return images, targets

ds_train_base = DentexDataset('train', YOLO_DIR, transform=BASE_TRANSFORM)
ds_val        = DentexDataset('val',   YOLO_DIR, transform=BASE_TRANSFORM)
ds_test       = DentexDataset('test',  YOLO_DIR, transform=BASE_TRANSFORM)

BATCH_SIZE = 8

dl_train_base = DataLoader(ds_train_base, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, collate_fn=collate_fn, pin_memory=True)
dl_val        = DataLoader(ds_val,        batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, collate_fn=collate_fn, pin_memory=True)
dl_test       = DataLoader(ds_test,       batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, collate_fn=collate_fn, pin_memory=True)

log(f'DataLoaders creados:', 'OK')
log(f'  train: {len(dl_train_base)} batches x {BATCH_SIZE} = {len(ds_train_base)} imgs', 'DATA')
log(f'  val:   {len(dl_val)} batches x {BATCH_SIZE} = {len(ds_val)} imgs', 'DATA')
log(f'  test:  {len(dl_test)} batches x {BATCH_SIZE} = {len(ds_test)} imgs', 'DATA')

## Seccion 6 — Preprocesamiento y normalizacion

| Paso | Valor | Justificacion |
|------|-------|---------------|
| `Resize(640, 640)` | 640x640 px | Tamano estandar de YOLOv8 |
| `ToTensor()` | [0,1] float32 | Convierte PIL [0,255] a tensor |
| `Normalize(mean, std)` | ImageNet | El backbone de YOLOv8 fue preentrenado con estas estadisticas |

Las radiografias son escala de grises convertidas a RGB (3 canales identicos).  
Normalizar con ImageNet RGB es equivalente a normalizar el canal gris.

In [ ]:
sample_img_raw = ds_train_base.get_image_raw(0)
sample_tensor, sample_target = ds_train_base[0]

log('Verificacion de preprocesamiento:', 'DATA')
log(f'  PIL original:      {sample_img_raw.size}  modo={sample_img_raw.mode}', 'DATA')
log(f'  Tensor shape:      {tuple(sample_tensor.shape)}  (C x H x W)', 'DATA')
log(f'  Tensor dtype:      {sample_tensor.dtype}', 'DATA')
log(f'  Rango normalizado: [{sample_tensor.min():.3f}, {sample_tensor.max():.3f}]', 'DATA')
log(f'  Boxes en sample:   {len(sample_target["boxes"])}', 'DATA')

mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(sample_img_raw)
axes[0].set_title(f'PIL original — {sample_img_raw.size[0]}x{sample_img_raw.size[1]} px')
axes[0].axis('off')

img_denorm = (sample_tensor * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
axes[1].imshow(img_denorm)
axes[1].set_title('Tensor desnormalizado — 640x640 px')
if len(sample_target['boxes']) > 0:
    for box, lbl in zip(sample_target['boxes'], sample_target['labels']):
        xc, yc, bw, bh = box.tolist()
        x1 = (xc - bw/2) * 640
        y1 = (yc - bh/2) * 640
        color = '#FF4444' if lbl == 1 else '#44CC44'
        axes[1].add_patch(patches.Rectangle(
            (x1, y1), bw*640, bh*640,
            linewidth=2, edgecolor=color, facecolor='none'))
axes[1].axis('off')

plt.suptitle('Preprocesamiento: PIL original vs tensor normalizado')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'preprocesamiento.png'), dpi=100, bbox_inches='tight')
plt.show()
log('Guardado: preprocesamiento.png', 'OK')

## Seccion 7 — Data augmentation

| Transformacion | Parametro | Justificacion clinica |
|---|---|---|
| `RandomHorizontalFlip` | p=0.5 | El arco dental es simetrico — espejo horizontal es valido |
| `ColorJitter` | brightness=0.3, contrast=0.3 | Simula variabilidad de exposicion entre los 3 hospitales |
| `RandomAffine` | degrees=5, translate=(0.05, 0.05) | Variaciones leves de posicionamiento del paciente |
| `RandomErasing` | p=0.1, scale=(0.02, 0.1) | Simula artefactos metalicos que ocluyen partes de la imagen |

**No aplicamos:**
- `RandomVerticalFlip` — anatomicamente incorrecto
- `RandomRotation` > 10 grados — las panoramicas tienen orientacion fija

Las augmentations se aplican **solo a train**. Val y test usan BASE_TRANSFORM.

In [ ]:
TRAIN_TRANSFORM = T.Compose([
    T.Resize((640, 640)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    T.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.1, scale=(0.02, 0.1), value='random')
])

ds_train = DentexDataset('train', YOLO_DIR, transform=TRAIN_TRANSFORM)
dl_train  = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, collate_fn=collate_fn, pin_memory=True)
log(f'Dataset train con augmentation: {len(ds_train)} imagenes', 'OK')

In [ ]:
# Visualizacion: 4 imagenes x 4 versiones aumentadas
n_imgs     = 4
n_versions = 4
indices    = random.sample(range(len(ds_train)), n_imgs)

fig, axes = plt.subplots(n_imgs, n_versions + 1, figsize=(22, 4 * n_imgs))

def denorm(t):
    return (t * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()

for row, idx in enumerate(indices):
    img_raw = ds_train.get_image_raw(idx)
    axes[row][0].imshow(img_raw.resize((640, 640)))
    axes[row][0].set_title('Original' if row == 0 else '', fontsize=9)
    axes[row][0].axis('off')
    for col in range(n_versions):
        aug_tensor, _ = ds_train[idx]
        axes[row][col + 1].imshow(denorm(aug_tensor))
        axes[row][col + 1].set_title(f'Aug {col+1}' if row == 0 else '', fontsize=9)
        axes[row][col + 1].axis('off')

plt.suptitle('Data Augmentation — misma imagen, distintas versiones\n'
             'Flip horizontal | ColorJitter | RandomAffine | RandomErasing')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'augmentation_ejemplos.png'), dpi=100, bbox_inches='tight')
plt.show()
log('Guardado: augmentation_ejemplos.png', 'OK')

In [ ]:
# Augmentation adicional: division en cuadrantes (SAHI-style)
log('Visualizando division en cuadrantes...', 'INFO')

sample_raw = ds_train.get_image_raw(indices[0])
W, H = sample_raw.size

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0][0].imshow(sample_raw)
axes[0][0].set_title(f'Imagen completa ({W}x{H})')
axes[0][0].axis('off')

cuadrantes = [
    ('Superior izq', (0, 0, W//2, H//2)),
    ('Superior der', (W//2, 0, W, H//2)),
    ('Inferior izq', (0, H//2, W//2, H)),
    ('Inferior der', (W//2, H//2, W, H)),
]
positions = [(0,1), (0,2), (1,0), (1,1)]
for (title, box), (r, c) in zip(cuadrantes, positions):
    tile = sample_raw.crop(box)
    axes[r][c].imshow(tile)
    axes[r][c].set_title(f'{title} ({tile.size[0]}x{tile.size[1]})', fontsize=9)
    axes[r][c].axis('off')

axes[1][2].imshow(sample_raw)
axes[1][2].axhline(y=H//2, color='yellow', linewidth=2)
axes[1][2].axvline(x=W//2, color='yellow', linewidth=2)
axes[1][2].set_title('Division en cuadrantes')
axes[1][2].axis('off')

plt.suptitle('Division en cuadrantes (SAHI) — mejora deteccion en imagenes anchas')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'augmentation_cuadrantes.png'), dpi=100, bbox_inches='tight')
plt.show()
log('Guardado: augmentation_cuadrantes.png', 'OK')

## Seccion 8 — Verificacion final del batch

Tomamos un batch real del DataLoader de train y verificamos dimensiones,  
rango de valores, ausencia de NaN/Inf, y correspondencia imagen-etiqueta.

In [ ]:
batch_imgs, batch_targets = next(iter(dl_train))

log('Verificacion del batch:', 'DATA')
log(f'  Shape:           {tuple(batch_imgs.shape)}  (B x C x H x W)', 'DATA')
log(f'  Dtype:           {batch_imgs.dtype}', 'DATA')
log(f'  Rango de valores: [{batch_imgs.min():.3f}, {batch_imgs.max():.3f}]', 'DATA')
log(f'  Boxes por imagen: {[len(t["boxes"]) for t in batch_targets]}', 'DATA')
log(f'  Labels por imagen: {[t["labels"].tolist() for t in batch_targets]}', 'DATA')

assert not torch.isnan(batch_imgs).any(), 'NaN detectado en el batch'
assert not torch.isinf(batch_imgs).any(), 'Inf detectado en el batch'
log('Sin NaN ni Inf en el batch', 'OK')

In [ ]:
# Visualizar el batch con boxes
n_show = min(8, len(batch_targets))
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for idx, ax in enumerate(axes.flatten()):
    if idx >= n_show:
        ax.axis('off')
        continue
    img_t  = batch_imgs[idx]
    target = batch_targets[idx]
    img_vis = (img_t * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_vis)
    for box, lbl in zip(target['boxes'], target['labels']):
        xc, yc, bw, bh = box.tolist()
        x1    = (xc - bw/2) * 640
        y1    = (yc - bh/2) * 640
        color = '#FF4444' if lbl == 1 else '#44CC44'
        label = 'imp' if lbl == 1 else 'eru'
        ax.add_patch(patches.Rectangle(
            (x1, y1), bw*640, bh*640,
            linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1+2, y1-8, label, color='white', fontsize=7, fontweight='bold',
                bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))
    ax.set_title(f'{target["image_id"]}  {len(target["boxes"])} boxes', fontsize=7)
    ax.axis('off')

plt.suptitle('Batch de train verificado — desnormalizado con boxes\nVerde=erupted  Rojo=impacted')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'batch_verificacion.png'), dpi=100, bbox_inches='tight')
plt.show()
log('Guardado: batch_verificacion.png', 'OK')

## Seccion 9 — CSVs de splits para GitHub

Los CSVs se versionan en el repositorio para garantizar reproducibilidad.  
Contienen las rutas relativas de cada imagen y su etiqueta.

In [ ]:
for split in ['train', 'val', 'test']:
    img_dir = YOLO_DIR / 'images' / split
    lbl_dir = YOLO_DIR / 'labels' / split
    rows = []
    for img_path in sorted(list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg'))):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        classes = []
        for line in lbl_path.read_text().strip().split('\n'):
            if line:
                classes.append(int(line.split()[0]))
        rows.append({
            'filename':   img_path.name,
            'split':      split,
            'img_class':  'impacted' if 1 in classes else 'erupted',
            'n_boxes':    len(classes),
            'n_erupted':  classes.count(0),
            'n_impacted': classes.count(1),
            'lbl_path':   f'labels/{split}/{lbl_path.name}'
        })
    df = pd.DataFrame(rows)
    csv_path = SPLITS_DIR / f'{split}.csv'
    df.to_csv(csv_path, index=False)
    log(f'CSV {split}: {len(df)} filas guardado en {csv_path}', 'OK')
    log(f'  erupted={( df.img_class=="erupted").sum()}  impacted={(df.img_class=="impacted").sum()}', 'DATA')

log('CSVs listos para subir a GitHub', 'OK')
log('Ubicacion: data/train.csv  data/val.csv  data/test.csv', 'INFO')

## Seccion 10 — Estrategias de entrenamiento evaluadas

### Opcion A — YOLOv8s fine-tuning desde COCO — ELEGIDA

```python
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(
    data='data/processed/yolo_dataset/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    patience=20,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
)
```

**Justificacion:**
- Preentrenado en COCO (80 clases, 118k imagenes) — representaciones visuales ricas
- Fine-tuning converge rapido con pocos datos (~440 imagenes)
- Deteccion + clasificacion en una sola pasada
- Metricas COCO nativas (AP50, AP75, mAP)
- API simple y bien documentada

---

### Opcion B — HierarchicalDet (metodo del paper) — DESCARTADA

El paper usa DiffusionDet con aprendizaje jerarquico en 3 etapas sobre los datasets (a), (b) y (c).

**Por que no se implementa:**
- Requiere **detectron2** — instalacion compleja, incompatible con versiones actuales de PyTorch en Colab
- DiffusionDet es un proceso iterativo de difusion — inferencia lenta (~10 segundos por imagen)
- Disenado para la tarea completa (3 jerarquias: cuadrante + enumeracion + diagnostico), no para deteccion binaria
- Requiere los 3 datasets para el entrenamiento jerarquico
- No hay script de fine-tuning simplificado — requiere adaptar todo el pipeline de detectron2

**Lo que tomamos del paper:**
- Metricas de evaluacion: AP50, AP75, mAP (COCO-style)
- Confirmacion de que fine-tuning funciona bien con este dataset
- Referencia de resultados para comparar

---

### Opcion C — SAHI (division en cuadrantes) — A EVALUAR POST-ENTRENAMIENTO

Las radiografias panoramicas son muy anchas (~2800px). Al redimensionar a 640px las muelas quedan pequenas (~30-50px). SAHI divide la imagen en tiles solapados y combina las detecciones.

```bash
pip install sahi
```

```python
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

model = AutoDetectionModel.from_pretrained('ultralytics', model_path='best.pt')
result = get_sliced_prediction(
    image,
    model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)
```

Se evaluara si las metricas del entrenamiento base no son satisfactorias.

In [ ]:
log('=' * 50, 'INFO')
log('RESUMEN — Entrega 2', 'INFO')
log('=' * 50, 'INFO')
log(f'Dataset:           DENTEX (cat2==7)', 'DATA')
log(f'Imagenes totales:  {len(ds_train)+len(ds_val)+len(ds_test)}', 'DATA')
log(f'  train: {len(ds_train)}  val: {len(ds_val)}  test: {len(ds_test)}', 'DATA')
log(f'Input size:        640x640 px', 'DATA')
log(f'Normalizacion:     ImageNet mean/std', 'DATA')
log(f'Augmentation:      HFlip + ColorJitter + Affine + RandomErasing', 'DATA')
log(f'Batch size:        {BATCH_SIZE}', 'DATA')
log(f'Modelo elegido:    YOLOv8s fine-tuning desde COCO', 'DATA')
log(f'Outputs:           {OUTPUTS_DIR}', 'OK')
log(f'CSVs splits:       {SPLITS_DIR}', 'OK')